In [7]:
import importlib.util
import sys
from pathlib import Path

SCHEMAS_DIR = Path("/home/hello/Projects/Statements/code/moltie/schemas")
assert SCHEMAS_DIR.exists(), f"Missing schemas dir: {SCHEMAS_DIR}"

PKG = "schemas"  # pretend package name

# Create an empty package module object for `schemas`
if PKG not in sys.modules:
    pkg_spec = importlib.util.spec_from_loader(PKG, loader=None)
    pkg_mod = importlib.util.module_from_spec(pkg_spec)
    pkg_mod.__path__ = [str(SCHEMAS_DIR)]  # mark as package
    sys.modules[PKG] = pkg_mod

def import_as_pkg(mod_name: str, file_path: Path):
    full_name = f"{PKG}.{mod_name}"
    spec = importlib.util.spec_from_file_location(full_name, str(file_path))
    assert spec and spec.loader, f"Cannot load spec for {file_path}"
    mod = importlib.util.module_from_spec(spec)
    sys.modules[full_name] = mod
    spec.loader.exec_module(mod)
    return mod

query_object  = import_as_pkg("query_object",  SCHEMAS_DIR / "query_object.py")
verdict       = import_as_pkg("verdict",       SCHEMAS_DIR / "verdict.py")
negative_exit = import_as_pkg("negative_exit", SCHEMAS_DIR / "negative_exit.py")
run_config    = import_as_pkg("run_config",    SCHEMAS_DIR / "run_config.py")

QueryObject  = query_object.QueryObject
AtomQuery    = query_object.AtomQuery
Verdict      = verdict.Verdict
Anchor       = verdict.Anchor
NegativeExit = negative_exit.NegativeExit
RunConfig    = run_config.RunConfig

print("✅ Loaded schemas as an in-memory package namespace")


✅ Loaded schemas as an in-memory package namespace


In [8]:
from pathlib import Path
import PyPDF2

appeal_dir = Path("/media/hello/Vault/Tribunals/EAT_Appeals/")
appeal_name = "Z_v_A_UKEAT_0203_13_SM_.pdf" # <-- change this

assert appeal_dir.exists(), f"Missing dir: {appeal_dir}"


PDF_PATH = appeal_dir / appeal_name  # <-- change this
assert PDF_PATH.exists(), f"Missing PDF: {PDF_PATH}"

def pdf_to_text(path: Path) -> str:
    out = []
    with path.open("rb") as f:
        reader = PyPDF2.PdfReader(f)
        for i, page in enumerate(reader.pages):
            txt = page.extract_text() or ""
            out.append(txt)
    return "\n".join(out)

doc_text = pdf_to_text(PDF_PATH)
print("PDF chars:", len(doc_text))
print("Preview:\n", doc_text[:1200])


PDF chars: 52945
Preview:
  Copyright 2013  Appeal No.  UKEAT /0203/13/SM  
    & UKEAT/0380/13/SM  
 
 
EMPLOYMENT APPEAL TRIBUNAL  
FLEETBANK HOUSE, 2 -6 SALISBURY SQUARE, LONDON, EC4Y 8JX  
 
 
 At the Tribunal  
 on 12th November 2013  
    Judgment handed down o n 9th December 2013  
 
 
Before  
THE HONOURABLE MR JUSTI CE LANGSTAFF  (PRESIDENT)  
MR I EZEKIEL  
MR H SINGH  
 
  
UKEAT/0203/13/SM  
 
Z  APPELLANT  
 
 
 
 
 
A RESPONDENT  
 
  
UKEAT/0380/13/SM  
 
A APPELLANT  
 
 
 
 
 
Z 
 RESPONDENT  
 
 
 
JUDGMENT  
 
 
UKEAT/0203/13/SM  
UKEAT/0308/13/SM  
   
 
 
 
 
 
 
 APPEARANCES  
 
 
 
 
 
For the Appellant  
(in UKEAT/0203/13/SM  
and the Respondent in  
UKEAT/0380/13/SM)  
 MR ANDREW WATSON  
(Representative)  
Instructed by:  
Free Representation Unit  
Ground Floor  
60 Gray's Inn Road  
London  
WC1X 8LU  
 
For the Respondent  
(in UKEAT/0203/13/SM  
and the Appellant in  
UKEAT/0380/13/SM)  
 MR BRUCE GARDINER  
 (of Counsel)  
Instructed by:  
Legal Services

In [9]:
import re
from dataclasses import dataclass
from typing import List, Tuple

QUERY_TEXT = """
appeal treated as final
predetermined
open-minded review
allegation presented as established conclusion
unauthorised disclosure
"""  # <-- paste your query here (WS-derived, or X indicators)

# ---- simple paragraph splitter ----
PARA_SPLIT = re.compile(r"\n\s*\n+")
WS_RE = re.compile(r"[ \t]+")

def clean_para(s: str) -> str:
    s = s.strip()
    s = WS_RE.sub(" ", s)
    return s.strip()

@dataclass(frozen=True)
class Para:
    para_id: str
    text: str

def to_paras(text: str) -> List[Para]:
    parts = PARA_SPLIT.split(text or "")
    paras = []
    n = 0
    for p in parts:
        p = clean_para(p)
        if len(p) < 20:
            continue
        n += 1
        paras.append(Para(para_id=f"p{n:05d}", text=p))
    return paras

paras = to_paras(doc_text)
print("Paras:", len(paras))

# ---- dumb keyword scoring ----
terms = [t.strip().lower() for t in QUERY_TEXT.splitlines() if t.strip()]
def score_para(p: Para) -> int:
    t = p.text.lower()
    return sum(1 for term in terms if term in t)

scored: List[Tuple[int, Para]] = [(score_para(p), p) for p in paras]
scored = [(s,p) for s,p in scored if s > 0]
scored.sort(key=lambda x: x[0], reverse=True)

print("Matched paras:", len(scored))
print("\n=== Top 10 matches ===")
for s, p in scored[:10]:
    print(f"\n[{p.para_id}] score={s}")
    print(p.text[:800], "..." if len(p.text) > 800 else "")


Paras: 89
Matched paras: 0

=== Top 10 matches ===


In [15]:
# === Moltie LLM verifier: single-document debug cell ===
# What this does:
# 1) Load ONE PDF -> text
# 2) Split into paragraph units with stable para_id
# 3) Build a minimal AtomQuery (manual for debug)
# 4) Build verifier prompt
# 5) Call Ollama and return a validated Verdict

from pathlib import Path
import re
import sys

import PyPDF2

# -----------------------
# 0) Paths / imports
# -----------------------
# If your project is packaged (recommended):
#   /home/hello/Projects/Statements/code/precedent_moltie/...
# then add its parent to sys.path and import normally.

PROJECT_ROOT = Path("/home/hello/Projects/Statements/code")  # adjust if needed
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Change `precedent_moltie` below if your package name differs

from moltie.schemas.query_object import AtomQuery
from moltie.schemas.run_config import RunConfig
from moltie.llm.verifier_prompt import build_verifier_prompt
from moltie.llm.client import verify_with_ollama, LLMClientConfig

# -----------------------
# 1) Load one PDF -> text
# -----------------------
appeal_dir = Path("/media/hello/Vault/Tribunals/EAT_Appeals/")
appeal_name = "Z_v_A_UKEAT_0203_13_SM_.pdf" # <-- change this

PDF_PATH = appeal_dir / appeal_name  # <-- set this

assert PDF_PATH.exists(), f"Missing PDF: {PDF_PATH}"

def pdf_to_text(path: Path) -> str:
    out = []
    with path.open("rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            out.append(page.extract_text() or "")
    return "\n".join(out)

doc_text = pdf_to_text(PDF_PATH)
print("PDF chars:", len(doc_text))
print("Preview:\n", doc_text[:800])

# -----------------------
# 2) Paragraph split + IDs
# -----------------------
PARA_SPLIT = re.compile(r"\n\s*\n+")
WS_RE = re.compile(r"[ \t]+")

def clean_para(s: str) -> str:
    s = s.strip()
    s = WS_RE.sub(" ", s)
    return s.strip()

def to_paras(text: str, min_len: int = 40):
    parts = PARA_SPLIT.split(text or "")
    paras = []
    n = 0
    for p in parts:
        p = clean_para(p)
        if len(p) < min_len:
            continue
        n += 1
        paras.append({"para_id": f"p{n:05d}", "text": p})
    return paras

paras = to_paras(doc_text)
print("Paras:", len(paras))
assert paras, "No paragraphs extracted — PDF text extraction likely failed."

# -----------------------
# 3) Evidence pack (top-K paras for debug)
# -----------------------
# For debugging, we just take the first K paras.
# Later retrieval will pick top paras.
# === Build evidence pack by keyword hits (cheap recall) ===

terms = [t.strip().lower() for t in atom.positive_indicators if t.strip()]
assert terms, "No terms in atom.positive_indicators"

def para_score(p):
    t = p["text"].lower()
    return sum(1 for term in terms if term and term in t)

scored = [(para_score(p), p) for p in paras]
scored = [(s, p) for s, p in scored if s > 0]
scored.sort(key=lambda x: x[0], reverse=True)

print("Matched paras:", len(scored))
print("Top scores:", [s for s,_ in scored[:10]])

K_DEBUG_PARAS = 20
top_paras = [p for _, p in scored[:K_DEBUG_PARAS]]

# If nothing matched, fall back to middle-of-doc paras (often where reasoning starts)
if not top_paras:
    mid = len(paras) // 2
    top_paras = paras[max(0, mid-10): mid+10]
    print("No keyword matches; using mid-doc fallback paras:", len(top_paras))

evidence_pack = {
    "doc_id": PDF_PATH.stem,
    "doc_meta": {"source_path": str(PDF_PATH)},
    "paras": top_paras,
    "retrieval": {"method": "keyword_hits", "score": None},
}

print("Evidence pack paras:", len(evidence_pack["paras"]))
print("\n--- Evidence pack preview (para_id, score, snippet) ---")
for s, p in scored[:5]:
    print(p["para_id"], "score=", s, "|", p["text"][:160].replace("\n"," "), "...")


# -----------------------
# 4) AtomQuery (manual for debug)
# -----------------------
# Replace indicators with your real ones (from Y_JSON -> QueryObject builder)
atom = AtomQuery(
    atom_id="X_DEBUG",
    x_tests=["X1","X2","X3","X4"],
    proposition="Appeal integrity undermined by predetermination / allegations treated as fact while appeal live / confidentiality breach",
    positive_indicators=[
        "appeal treated as final",
        "predetermined",
        "open-minded review",
        "allegation presented as an established conclusion",
        "unauthorised disclosure",
        "confidentiality",
    ],
    excludes=[],
    keyword_seeds=[],
    expansion_terms=[],
)

cfg = RunConfig(
    anchors_required=2,
    thresh_score=70,
    thresh_conf=75,
)

# -----------------------
# 5) Build prompt + call Ollama
# -----------------------
prompt = build_verifier_prompt(atom=atom, evidence_pack=evidence_pack, cfg=cfg)

client_cfg = LLMClientConfig(
    model="mistral-small3.2:latest",                 # set your Ollama model
    ollama_url="http://localhost:11434/api/generate",# set if different
    timeout_s=180,
    temperature=0.0,
    num_predict=800,
    max_retries=2,
)

verdict = verify_with_ollama(prompt, client_cfg)

# -----------------------
# 6) Inspect result
# -----------------------
print("\n=== VERDICT ===")
print(verdict.to_dict())

print("\n=== ANCHORS (verbatim) ===")
for a in verdict.anchors:
    print(f"- {a.para_id}: {a.quote[:180]}{'...' if len(a.quote)>180 else ''}")
    print(f"  why: {a.why_it_matters}")


PDF chars: 52945
Preview:
  Copyright 2013  Appeal No.  UKEAT /0203/13/SM  
    & UKEAT/0380/13/SM  
 
 
EMPLOYMENT APPEAL TRIBUNAL  
FLEETBANK HOUSE, 2 -6 SALISBURY SQUARE, LONDON, EC4Y 8JX  
 
 
 At the Tribunal  
 on 12th November 2013  
    Judgment handed down o n 9th December 2013  
 
 
Before  
THE HONOURABLE MR JUSTI CE LANGSTAFF  (PRESIDENT)  
MR I EZEKIEL  
MR H SINGH  
 
  
UKEAT/0203/13/SM  
 
Z  APPELLANT  
 
 
 
 
 
A RESPONDENT  
 
  
UKEAT/0380/13/SM  
 
A APPELLANT  
 
 
 
 
 
Z 
 RESPONDENT  
 
 
 
JUDGMENT  
 
 
UKEAT/0203/13/SM  
UKEAT/0308/13/SM  
   
 
 
 
 
 
 
 APPEARANCES  
 
 
 
 
 
For the Appellant  
(in UKEAT/0203/13/SM  
and the Respondent in  
UKEAT/0380/13/SM)  
 MR ANDREW WATSON  
(Representative)  
Instructed by:  
Free Representation Unit  
Ground Floor  
60 Gray's Inn Road  
Lond
Paras: 88
Matched paras: 0
Top scores: []
No keyword matches; using mid-doc fallback paras: 20
Evidence pack paras: 20

--- Evidence pack preview (para_id, score, snippet) 

RuntimeError: verify_with_ollama failed after 3 attempts. Last error: Could not locate JSON object braces in model output. Last output (truncated): "The document discusses an appeal case regarding the fairness of an employee's dismissal due to allegations of sexual abuse. The key points are:\n\n1. **Legal Principles**: The case references earlier judgments (A v B and Leach v Ofcom) which establish that an employer must not accept allegations uncritically, even if they come from an authoritative body. The decision to dismiss based on such allegations must be assessed on a case-by-case basis.\n\n2. **Factual Differences**: The current case differs from A v B and Leach because the employee worked directly with children, and the police did not endorse the allegations as their own view at the time of dismissal.\n\n3. **Tribunal's Role**: The Employment Tribunal must examine all relevant circumstances to determine if the reason for dismissal is substantial and sufficient. Their decision is a